<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Counting, and tracking in brief

**[Counting, and tracking in brief](https://learning.nextia-ai.com/courses/vision/m04/counting-and-tracking-in-brief/)** · Vision AI: See, Count and Inspect · Module 4, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** count boxes on a pallet in three ways — from a detector's boxes, by a rule, and with a density estimate — and see where crowding breaks each one.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells and the model (47 MB). No account and no key. |
| **You should know** | Boxes, IoU and NMS, from [Boxes and IoU](https://learning.nextia-ai.com/courses/vision/m04/boxes-and-iou/); the fine-tuned detector, from [Fine-tune a detector on a few images](https://learning.nextia-ai.com/courses/vision/m04/fine-tune-a-detector-on-a-few-images/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m04/counting-and-tracking-in-brief/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M04-L04-counting-and-tracking-in-brief/counting-and-tracking-in-brief-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 6 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/pallets.zip": "cec4b211fa4e434936fc0c1980accfc74392f1ced9de6c04b5421b9ec6af56e3",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The pallets

The count is the boxes you can see: a box behind another box is not counted. Each test photo is loose, medium or tight.

In [ ]:
def load(split):
    coco = json.load(open(f"data/pallets/{split}.json"))
    for im in coco["images"]:
        im["boxes"] = [a["bbox"] for a in coco["annotations"] if a["image_id"] == im["id"]]
    return coco["images"]

train, test = load("train"), load("test")
truth = [im["count"] for im in test]
crowd = [im["crowding"] for im in test]
print(pd.Series(truth).groupby(pd.Series(crowd)).describe()[["count", "min", "max"]])

## 2. Way 1: count the detector's boxes

The recorded boxes of the detector fine-tuned on 40 photos (last lesson). Its score line, 0.6, was chosen on the validation photos.

In [ ]:
rec = json.load(open("data/predictions/pallets_fasterrcnn_finetuned_40.json"))

def box_counts(score, nms_line=None):
    out = []
    for im in test:
        P = [p for p in rec if p["scene_id"] == im["scene_id"] and p["score"] >= score]
        if nms_line is not None:
            P = [P[i] for i in cp.nms([p["box"] for p in P], [p["score"] for p in P], nms_line)]
        out.append(len(P))
    return out

def report(name, counts):
    s = cp.count_scores(truth, counts, crowd)
    print(f"{name:32s} error {s['mae']:5} | " + " ".join(f"{k} {v['mae']}" for k, v in s["by_crowding"].items()))

report("boxes, score 0.6", box_counts(0.6))
report("boxes, score 0.5", box_counts(0.5))

## 3. Way 2: a rule with no model

7 boxes in 10 carry a white label. Count the white blobs and divide by 0.7. It takes a few milliseconds a photo.

> **Predict.** Will the rule be worse on tight pallets or on loose ones?

In [ ]:
from scipy import ndimage

def label_rule(path):
    """Count the white labels and divide by 0.7 (7 boxes in 10 have a label)."""
    a = np.asarray(Image.open(path).convert("RGB"), float)
    white = a.min(2) > 200                      # very light pixels in all three channels
    lab, n = ndimage.label(white)               # group touching white pixels into blobs
    sizes = ndimage.sum(white, lab, range(1, n + 1))
    return int(round((np.asarray(sizes) >= 12).sum() / 0.7))   # blobs of 12 pixels or more

rule = [label_rule("data/" + im["file_name"]) for im in test]
report("rule: labels / 0.7", rule)

> **Check.** The rule's error is 5.23 boxes a photo, and it is worst on **loose** pallets: crowding does not break it, but a pallet with few labels does.

## 4. Way 3: a density estimate

A density estimate does not find boxes one by one. It cuts the photo into cells and predicts how many box centres fall in each cell; the count is the sum. We take ResNet-18 features for each 16 × 16 pixel cell and fit a ridge regression on the 40 training photos. This takes about 20 seconds.

In [ ]:
import torch, timm
from huggingface_hub import hf_hub_download
from sklearn.linear_model import Ridge
torch.set_num_threads(4)
weights = hf_hub_download("timm/resnet18.a1_in1k", "model.safetensors", revision="491b427b45c94c7fb0e78b5474cc919aff584bbf")
backbone = timm.create_model("resnet18.a1_in1k", pretrained=True, features_only=True, out_indices=(3,),
                             pretrained_cfg_overlay={"file": weights}).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def cells(path):
    """256 numbers for each 16 x 16 pixel cell of the photo: 24 x 32 cells."""
    x = torch.from_numpy(np.asarray(Image.open(path).convert("RGB"), np.float32) / 255).permute(2, 0, 1)
    with torch.no_grad():
        f = backbone(((x - MEAN) / STD)[None])[0][0]
    return f.permute(1, 2, 0).reshape(-1, f.shape[0]).numpy(), tuple(f.shape[1:])

def centres(im, shape):
    """How many true box centres fall in each cell: the target of the regression."""
    g = np.zeros(shape)
    for b in im["boxes"]:
        cy, cx = (b[1] + b[3] / 2) / 16, (b[0] + b[2] / 2) / 16
        g[min(int(cy), shape[0] - 1), min(int(cx), shape[1] - 1)] += 1
    return g.ravel()

X, Y = [], []
for im in train:
    f, shape = cells("data/" + im["file_name"])
    X.append(f); Y.append(centres(im, shape))
reg = Ridge(alpha=10.0).fit(np.vstack(X), np.concatenate(Y))
density = [round(float(reg.predict(cells("data/" + im["file_name"])[0]).sum()), 1) for im in test]
report("density estimate", density)

## 5. All three on one chart

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, c, m in (("boxes, score 0.6", box_counts(0.6), "o"), ("rule", rule, "s"), ("density", density, "^")):
    ax.scatter(truth, c, label=name, marker=m, alpha=0.7)
ax.plot([0, 60], [0, 60], color="grey", lw=1, ls="--")
ax.set_xlabel("true count"); ax.set_ylabel("method's count"); ax.legend(); plt.show()

## 6. Your turn: the density estimate on tight pallets

> **Your turn.** Put the density estimate's mean count error on the 10 **tight** pallets in `density_tight` (use `cp.count_scores`, or compute it yourself).

In [ ]:
density_tight = cp.count_scores(truth, density, crowd)["by_crowding"]["tight"]["mae"]
print(density_tight)

In [ ]:
expect('density_tight', density_tight, 3.76, tolerance=0.02)

## 7. Failure case: a right count with wrong boxes

A count error of 0 does not mean the boxes are right. On the tight pallet S-063, the detector at score 0.6 keeps 22 boxes for 22 true boxes, but one of them is wrong and one true box is missed: the two errors cancel. Check the boxes on a sample of photos, not only the count.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m04/counting-and-tracking-in-brief/) compares the three ways by cost, and explains tracking across video frames.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Semantic and instance segmentation](https://learning.nextia-ai.com/courses/vision/m05/semantic-and-instance-segmentation/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m04/counting-and-tracking-in-brief/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.